# CP1 Week 12 -- File I/O: Reading & Writing

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Read and write text files with `open()`
2. Use the `csv` module to read/write CSV files
3. Use `json` module to read/write JSON
4. Implement `export_results()` for your pipeline

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: How Files Work

Until now, all your data has been defined directly in Python code.
In the real world, data comes from **files** on disk:

```
  Your program           Disk
  +------------+         +-------------------+
  | data = []  |  <---   | data/raw/data.csv |
  | clean...   |         | reports/report.json|
  | analyze... |  --->   | reports/figures/   |
  +------------+         +-------------------+
       RAM                    Permanent storage
```

**Key concept:** Variables in RAM disappear when your program ends.
Files on disk persist forever (until deleted). That is why we need file I/O.

### The `with open()` pattern

```python
with open("path/to/file.txt", "r") as f:   # "r" = read
    content = f.read()

with open("path/to/file.txt", "w") as f:   # "w" = write (overwrites!)
    f.write("Hello!")
```

The `with` keyword ensures the file is properly closed, even if an error occurs.

---
## Part 2: Reading CSV Files

In [ ]:
import csv
import os

# Create sample data
os.makedirs("data/raw", exist_ok=True)
with open("data/raw/sample.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "timestamp", "value", "status"])
    writer.writerow([1, "2024-01-01", "25.3", "ok"])
    writer.writerow([2, "2024-01-02", "88.1", "warning"])
    writer.writerow([3, "2024-01-03", "", "error"])
    writer.writerow([4, "2024-01-04", "42.0", "ok"])
print("Created sample.csv")

# Read it back
with open("data/raw/sample.csv", "r") as f:
    reader = csv.DictReader(f)
    data = list(reader)

for row in data:
    print(f"  {row}")

**Expected Output:**
```
{'id': '1', 'timestamp': '2024-01-01', 'value': '25.3', 'status': 'ok'}
{'id': '2', 'timestamp': '2024-01-02', 'value': '88.1', 'status': 'warning'}
{'id': '3', 'timestamp': '2024-01-03', 'value': '', 'status': 'error'}
{'id': '4', 'timestamp': '2024-01-04', 'value': '42.0', 'status': 'ok'}
```

Notice: `csv.DictReader` automatically uses the first row as column names.
Every value comes in as a **string** -- you must convert to float/int if needed.

### Try It Yourself

In [ ]:
# TODO: Read the CSV and compute the average of valid values
# Steps:
# 1. Read data/raw/sample.csv using csv.DictReader
# 2. Extract the "value" column
# 3. Skip empty or non-numeric values
# 4. Compute and print the average

# Your code here:


---
## Part 3: Writing CSV and JSON

In [ ]:
import json

# Write cleaned CSV
os.makedirs("data/cleaned", exist_ok=True)
clean = [row for row in data if row["value"] and row["status"] == "ok"]

with open("data/cleaned/cleaned.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "timestamp", "value", "status"])
    writer.writeheader()
    writer.writerows(clean)
print(f"Wrote {len(clean)} rows to cleaned.csv")

# Write JSON report
os.makedirs("reports", exist_ok=True)
report = {
    "project_name": "my_project",
    "track": "data",
    "version": "v1",
    "dataset": {"n_raw": len(data), "n_clean": len(clean), "n_dropped": len(data) - len(clean)},
    "analysis_summary": {"mean": 33.65, "min": 25.3, "max": 42.0},
    "figures": ["timeseries.png", "summary.png"],
}

with open("reports/report.json", "w") as f:
    json.dump(report, f, indent=2)
print("Wrote report.json")

---
## Part 3: Building export_results()

In [ ]:
def export_results(clean_data, results, figures, config):
    """Export all pipeline outputs."""
    exported = {}

    # Export cleaned CSV
    csv_path = config.get("cleaned_data_path", "data/cleaned/cleaned.csv")
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    if clean_data:
        keys = list(clean_data[0].keys())
        with open(csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(clean_data)
        exported["cleaned_csv"] = csv_path
        print(f"Exported: {csv_path}")

    # Export report.json
    report_path = config.get("report_path", "reports/report.json")
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    with open(report_path, "w") as f:
        json.dump(results, f, indent=2)
    exported["report"] = report_path
    print(f"Exported: {report_path}")

    return exported

exported = export_results(
    clean, report, [],
    {"cleaned_data_path": "data/cleaned/cleaned.csv", "report_path": "reports/report.json"}
)
print(f"\nExported: {exported}")

### Part 4: Reading JSON back

In [ ]:
# Read the report back and display it
import json

with open("reports/report.json", "r") as f:
    loaded_report = json.load(f)

print("=== Loaded Report ===")
for key, value in loaded_report.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k2, v2 in value.items():
            print(f"    {k2}: {v2}")
    else:
        print(f"  {key}: {value}")

### Common Mistakes with File I/O

| Mistake | What happens | Fix |
|---------|-------------|-----|
| Forget `newline=""` in csv.writer | Extra blank lines on Windows | Always use `newline=""` |
| Forget to create directories | FileNotFoundError | Use `os.makedirs(..., exist_ok=True)` |
| Open file without `with` | File may not close properly | Always use `with open(...)` |
| Write dict to CSV without DictWriter | TypeError | Use `csv.DictWriter` for dicts |
| Forget `indent` in json.dump | One long line, hard to read | Use `json.dump(data, f, indent=2)` |

### Debugging Tip

If you see `FileNotFoundError`, the directory does not exist. Always run
`os.makedirs()` before writing to a new path.

### Why This Matters for Your Pipeline

File I/O is the **output** of your pipeline. Without it, all your analysis
results disappear when the program ends. `export_results()` makes your work
permanent and shareable:
- **cleaned.csv** -- can be loaded into Excel, pandas, or another pipeline
- **report.json** -- can be read by web dashboards or other programs
- **figures** -- can be included in reports and presentations

---
## Key Takeaways -- Week 12

1. **`csv.DictReader`** reads CSV into dicts; **`csv.DictWriter`** writes them
2. **`json.dump()`** writes Python dicts to JSON; **`json.load()`** reads them back
3. **Always use `os.makedirs()`** to create directories before writing
4. **`with open()`** ensures files are properly closed
5. **`export_results()`** writes both cleaned data and report

### Try It Yourself

In [ ]:
# TODO: Write a complete export_results() function that:
# 1. Writes cleaned data to a CSV file
# 2. Writes a report dict to a JSON file
# 3. Returns a dict of exported file paths
# 4. Prints what was exported

def my_export_results(clean_data, results, config):
    exported = {}
    # Your code here
    return exported

# Test with sample data
test_clean = [{"id": 1, "value": 25}, {"id": 2, "value": 50}]
test_results = {"project": "test", "mean": 37.5}
test_config = {
    "cleaned_data_path": "data/cleaned/test_cleaned.csv",
    "report_path": "reports/test_report.json",
}
# my_export_results(test_clean, test_results, test_config)

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What does csv.DictReader do?
# R2: What does json.dump() do? What about json.load()?
# R3: Why is 'with open()' better than just 'open()'?
# R4: What does newline="" do in csv.writer?

### Practice (P1-P5)

In [ ]:
# P1: Write load_csv(path) that reads any CSV and returns a list of dicts.


In [ ]:
# P2: Write save_csv(data, path) that writes a list of dicts to CSV.


In [ ]:
# P3: Write a complete export_results() for your track.


In [ ]:
# P4: Read report.json back and print a formatted summary.


In [ ]:
# P5: Write a function that counts rows in a CSV without loading all into memory.


### Challenge (C1-C3)

In [ ]:
# C1: Write a backup system that saves timestamped copies.
# Example: report_2026-03-26_14-30.json


In [ ]:
# C2: Write a function that merges two CSV files with the same headers.


In [ ]:
# C3: Write a function that compares two report.json files and
# prints what changed between them.


### Mini-Project

In [ ]:
# M1: Complete File I/O System
# Write a full I/O module with:
# - load_csv(), save_csv()
# - load_json(), save_json()
# - export_results() that creates all required files
# - A verify function that reads everything back and checks it
# Test the entire flow: create data -> export -> reload -> verify


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)